In [ ]:
import pandas as pd
import numpy as np
import yaml
import logging
import datania

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ============================================
# TASK 1: Setup Components
# ============================================

# Create and save sample data
logger.info("Preparing sample data...")
qlfs_data = datania.generate_labour_force_survey(n_persons=1000, seed=42)
raw_df = pd.read_csv(qlfs_data)
raw_df.to_csv('pipeline_raw_qlfs.csv', index=False)

# Configuration
config_yaml = """
cleaning:
  min_age: 15
  max_age: 64
  required_columns:
    - employment_status
    - province
    - monthly_income

indicators:
  poverty_line: 2500
  income_column: monthly_income
"""

# Catalog
catalog_yaml = """
raw_qlfs:
  path: pipeline_raw_qlfs.csv
  format: csv
  description: Raw QLFS microdata
  read_only: true
  dtypes:
    person_id: str
    hh_id: str

clean_qlfs:
  path: pipeline_clean_qlfs.csv
  format: csv
  description: Cleaned QLFS data

provincial_indicators:
  path: pipeline_provincial_indicators.csv
  format: csv
  description: Final employment and income indicators by province
"""

# Parse configurations
config_dict = yaml.safe_load(config_yaml)
catalog_dict = yaml.safe_load(catalog_yaml)


# ============================================
# Simple Config and Catalog classes
# (You can reuse your code from previous exercises)
# ============================================

class Config:
    def __init__(self, config_dict):
        self._config = config_dict

    def get_cleaning(self):
        return self._config['cleaning']

    def get_indicators(self):
        return self._config['indicators']


class DataCatalog:
    def __init__(self, catalog_dict):
        self._catalog = catalog_dict

    def load(self, name):
        entry = self._catalog[name]
        dtypes = entry.get('dtypes', None)
        return pd.read_csv(entry['path'], dtype=dtypes)

    def save(self, df, name):
        entry = self._catalog[name]
        if entry.get('read_only', False):
            raise ValueError(f"Cannot save to read-only dataset: {name}")
        df.to_csv(entry['path'], index=False)


# ============================================
# TASK 2: Create Node Functions
# ============================================

def clean_survey_data(df: pd.DataFrame, params: dict) -> pd.DataFrame:
    """
    Clean survey data: filter to working age and drop missing values.

    Parameters:
        df: Raw survey DataFrame
        params: Dictionary with 'min_age', 'max_age', 'required_columns'

    Returns:
        Cleaned DataFrame
    """
    logger.info(f"Cleaning data: {len(df)} input rows")

    # YOUR CODE HERE:
    # 1. Filter to working age (min_age to max_age)
    # 2. Drop rows with missing values in required_columns
    # 3. Log how many rows remain
    # 4. Return cleaned DataFrame
    pass


def compute_provincial_indicators(df: pd.DataFrame, params: dict) -> pd.DataFrame:
    """
    Compute employment rate and average income by province.

    Parameters:
        df: Cleaned survey DataFrame
        params: Dictionary with 'income_column' and 'poverty_line'

    Returns:
        DataFrame with provincial indicators
    """
    logger.info("Computing provincial indicators...")

    # YOUR CODE HERE:
    # 1. Group by province
    # 2. Calculate: total_persons, employed_count, employment_rate
    # 3. Calculate: avg_income, below_poverty_count, poverty_rate
    # 4. Return DataFrame with all indicators
    #
    # Hints:
    # - Employment: df['employment_status'] == 'Employed'
    # - Poverty: df[income_column] < poverty_line
    pass


# ============================================
# TASK 3: Build the Orchestrator
# ============================================

def run_pipeline(config: Config, catalog: DataCatalog) -> pd.DataFrame:
    """
    Execute the QLFS analysis pipeline.

    Parameters:
        config: Configuration object
        catalog: Data catalog object

    Returns:
        Final indicators DataFrame
    """
    logger.info("="*50)
    logger.info("Starting QLFS Analysis Pipeline")
    logger.info("="*50)

    try:
        # Step 1: Load raw data
        logger.info("Step 1: Loading raw data...")
        # YOUR CODE HERE

        # Step 2: Clean data
        logger.info("Step 2: Cleaning data...")
        # YOUR CODE HERE

        # Step 3: Compute indicators
        logger.info("Step 3: Computing indicators...")
        # YOUR CODE HERE

        # Step 4: Save outputs
        logger.info("Step 4: Saving outputs...")
        # YOUR CODE HERE (save clean data and indicators)

        logger.info("="*50)
        logger.info("Pipeline completed successfully!")
        logger.info("="*50)

        return indicators

    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        raise


# ============================================
# TASK 4: Run the Pipeline
# ============================================

if __name__ == '__main__':
    # Initialize components
    config = Config(config_dict)
    catalog = DataCatalog(catalog_dict)

    # Run pipeline
    indicators = run_pipeline(config, catalog)

    # Display results
    print("\n" + "="*50)
    print("PROVINCIAL EMPLOYMENT INDICATORS")
    print("="*50)
    print(indicators.to_string(index=False))